# Task 3. Forecasting model design, training and evaluation

Thin orchestrator: load the area series, train SARIMA (Fourier-ARIMA dynamic
harmonic regression), an LSTM, and a dilated 1D CNN, then run a shared
walk-forward evaluation across the test week (Dec 16 to 22, 2013).

The full sweep across all three areas is run by `make train && make forecast && make eval`.
This notebook demonstrates the pipeline on a single area for narrative purposes.


In [ ]:
from datetime import datetime
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import Image, display

from mtraffic.config import Config
from mtraffic.data.loaders import load_area_series
from mtraffic.eval import metrics, walkforward
from mtraffic.eval.plots import plot_combined, plot_forecast
from mtraffic.models.classical.sarima import SarimaModel
from mtraffic.models.neural.cnn_torch import CnnConfig, CnnForecaster
from mtraffic.models.neural.lstm_torch import LstmConfig, LstmForecaster
from mtraffic.utils.seed import seed_all

seed_all(42)

cfg = Config.load()
AREA = 5161
TRAIN_END = cfg.eval.train_end
TEST_START = cfg.eval.test_start
TEST_END = cfg.eval.test_end

series = load_area_series(cfg.paths.interim_dir, AREA)
train_series = series.loc[:TRAIN_END]
print(f'Area {AREA}: {len(series):,} obs, train ends {TRAIN_END}, test {TEST_START} to {TEST_END}')

## SARIMA

ARIMA(2,0,2) endogenous plus K=4 daily and K=3 weekly Fourier exogenous regressors.
This is the dynamic harmonic regression approach from FPP3 chapter 9, chosen over a
seasonal state space (s=144) which is intractable on CPU for our train length.


In [ ]:
sarima = SarimaModel(order=(2, 0, 2), daily_terms=4, weekly_terms=3)
sarima.fit(train_series)
print('SARIMA fitted')

## LSTM

Two-layer LSTM, hidden size 64, sequence length 144 (one day). Time features
(sin/cos of time-of-day, day-of-week, is_weekend) concatenated with the standardised
traffic value.


In [ ]:
lstm_cfg = LstmConfig(
    seq_len=cfg.lstm.seq_len,
    hidden=cfg.lstm.hidden,
    layers=cfg.lstm.layers,
    epochs=cfg.lstm.epochs,
    batch_size=cfg.lstm.batch_size,
    lr=cfg.lstm.lr,
)
lstm = LstmForecaster(lstm_cfg)
lstm.fit(train_series)
print('LSTM fitted')

## Dilated 1D CNN (TCN-style)

Dilated causal convolutions with dilation set {1, 2, 4, 8, 16, 32}, kernel 3,
residual blocks. Receptive field reaches ~190 timesteps, enough to span a full day.


In [ ]:
cnn_cfg = CnnConfig(
    seq_len=cfg.cnn.seq_len,
    channels=cfg.cnn.channels,
    dilations=tuple(cfg.cnn.dilations),
    kernel_size=cfg.cnn.kernel_size,
    epochs=cfg.cnn.epochs,
    batch_size=cfg.cnn.batch_size,
    lr=cfg.cnn.lr,
)
cnn = CnnForecaster(cnn_cfg)
cnn.fit(train_series)
print('CNN fitted')

## Walk-forward evaluation

At each test timestamp t every model sees the actual history up to t (no leakage)
and emits a one-step-ahead forecast. The protocol is identical across models.


In [ ]:
results = {}
for name, model in [('SARIMA', sarima), ('LSTM', lstm), ('CNN', cnn)]:
    res = walkforward.run(model, series, TEST_START, TEST_END)
    results[name] = res
    m = metrics.compute_all(res.y_true, res.y_pred)
    print(f'{name:7s}  MAE {m["mae"]:8.2f}  RMSE {m["rmse"]:8.2f}  MAPE {m["mape"]:6.2f}%  sMAPE {m["smape"]:6.2f}%')

## Metrics table (persisted by `make eval`)


In [ ]:
tabs = cfg.paths.reports_dir / 'tables'
metrics_csv = tabs / 'task3_metrics_all.csv'
if metrics_csv.exists():
    pd.read_csv(metrics_csv)
else:
    print(f'{metrics_csv} not found. Run: make eval')

## Forecast figures


In [ ]:
figs = cfg.paths.reports_dir / 'figures'
combined = figs / f'forecast_{AREA}_combined.png'
if combined.exists():
    display(Image(combined))
else:
    print(f'{combined} not found. Run: make forecast')

## Failure analysis

`reports/tables/failure_cases.csv` lists windows where every model simultaneously
exceeded its own average MAE by 2x or more. Two of three areas show their worst
window on Sunday Dec 22, the last day before Christmas week.


In [ ]:
failure_csv = tabs / 'failure_cases.csv'
if failure_csv.exists():
    pd.read_csv(failure_csv)
else:
    print(f'{failure_csv} not found. Run: make eval')